In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import KFold
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, LeakyReLU
)
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import shutil
from pathlib import Path

# Clear session and collect garbage
tf.keras.backend.clear_session()
gc.collect()
print('Session cleared.')

# Set constants
IMG_SIZE = (299, 299)  
BATCH_SIZE = 16
NUM_FOLDS = 5
BASE_LR = 2e-4
WEIGHT_DECAY = 0.003  
DROPOUT_RATE = 0.3

# Dataset paths
dataset_dir = "dataset"
neutrophil_dir = os.path.join(dataset_dir, "Neutrophil")
non_neutrophil_dir = os.path.join(dataset_dir, "NonNeutrophil")

# Create temporary directories for k-fold
temp_dir = "temp_kfold"
os.makedirs(temp_dir, exist_ok=True)


def create_model():
    inception_base = InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(299, 299, 3)
    )

    # Unfreeze the top N layers of InceptionV3
    for layer in inception_base.layers[:-50]:
        layer.trainable = False
    for layer in inception_base.layers[-50:]:
        layer.trainable = True

    # Build model architecture
    inputs = Input(shape=(299, 299, 3))
    x = preprocess_input(inputs)

    # Pass through the InceptionV3 base model
    x = inception_base(x)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT_RATE)(x)


    # First Dense layer
    x = Dense(1024, kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT_RATE)(x)

    # # # Second Dense layer
    x = Dense(512, kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = LeakyReLU(alpha=0.1)(x)
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT_RATE)(x)

    # Third Dense layer
    x = Dense(256, kernel_regularizer=l2(WEIGHT_DECAY))(x)
    x = LeakyReLU(alpha=0.1)(x) 
    x = BatchNormalization()(x)
    x = Dropout(DROPOUT_RATE)(x)

    # Binary classification output
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model, inception_base

# Function to create augmentation model
def create_augmentation_model():
    return Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
        tf.keras.layers.RandomContrast(0.2),
        # Additional augmentation
        tf.keras.layers.RandomBrightness(0.3)
    ])

def prepare_fold_datasets(neutrophil_files, non_neutrophil_files, fold_idx, train_indices, val_indices):
    # Clear previous fold data
    for split in ['train', 'val']:
        for cls in ['Neutrophil', 'NonNeutrophil']:
            os.makedirs(os.path.join(temp_dir, split, cls), exist_ok=True)
            for f in os.listdir(os.path.join(temp_dir, split, cls)):
                os.remove(os.path.join(temp_dir, split, cls, f))

    # Copy files for training set
    for idx in train_indices:
        if idx < len(neutrophil_files):
            src = os.path.join(neutrophil_dir, neutrophil_files[idx])
            dst = os.path.join(temp_dir, 'train', 'Neutrophil', neutrophil_files[idx])
            shutil.copy(src, dst)
        else:
            adjusted_idx = idx - len(neutrophil_files)
            src = os.path.join(non_neutrophil_dir, non_neutrophil_files[adjusted_idx])
            dst = os.path.join(temp_dir, 'train', 'NonNeutrophil', non_neutrophil_files[adjusted_idx])
            shutil.copy(src, dst)

    # Copy files for validation set
    for idx in val_indices:
        if idx < len(neutrophil_files):
            src = os.path.join(neutrophil_dir, neutrophil_files[idx])
            dst = os.path.join(temp_dir, 'val', 'Neutrophil', neutrophil_files[idx])
            shutil.copy(src, dst)
        else:
            adjusted_idx = idx - len(neutrophil_files)
            src = os.path.join(non_neutrophil_dir, non_neutrophil_files[adjusted_idx])
            dst = os.path.join(temp_dir, 'val', 'NonNeutrophil', non_neutrophil_files[adjusted_idx])
            shutil.copy(src, dst)

    # Create datasets
    train_dataset = image_dataset_from_directory(
        directory=os.path.join(temp_dir, 'train'),
        label_mode="binary",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        class_names=["NonNeutrophil", "Neutrophil"]
    )

    val_dataset = image_dataset_from_directory(
        directory=os.path.join(temp_dir, 'val'),
        label_mode="binary",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        class_names=["NonNeutrophil", "Neutrophil"]
    )

    # Apply data augmentation to training set
    augmentation_model = create_augmentation_model()
    train_dataset = train_dataset.map(
        lambda x, y: (augmentation_model(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Optimize for performance
    train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
    val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

    return train_dataset, val_dataset


def train_model(model, train_dataset, val_dataset, fold_idx):
    # Calculate class weights
    total_samples = 0
    neutrophil_count = 0

    for images, labels in train_dataset:
        total_samples += labels.shape[0]
        neutrophil_count += int(tf.reduce_sum(labels))

    non_neutrophil_count = total_samples - neutrophil_count

    # Calculate balanced class weights
    if non_neutrophil_count > 0 and neutrophil_count > 0:
        weight_for_0 = (1 / non_neutrophil_count) * (total_samples / 2.0)
        weight_for_1 = (1 / neutrophil_count) * (total_samples / 2.0)
    else:
        weight_for_0 = 1.0
        weight_for_1 = 1.0

    class_weight_dict = {
        0: weight_for_0,
        1: weight_for_1
    }

    print(f"Fold {fold_idx+1} class weights:", class_weight_dict)

    # Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=40,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    )

    # Compile model (using Adam now)
    model.compile(
        optimizer = AdamW(learning_rate=BASE_LR, weight_decay=WEIGHT_DECAY),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    print(f"\nFold {fold_idx+1} - Training model (with partial fine-tuning)")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=150,
        class_weight=class_weight_dict,
        callbacks=[early_stopping, reduce_lr]
    )

    return history, model

def evaluate_model(model, val_dataset, fold_idx):
    # Evaluate on validation set
    print(f"\nFold {fold_idx+1} - Evaluating model")
    results = model.evaluate(val_dataset)
    print(f"Loss: {results[0]:.4f}, Accuracy: {results[1]:.4f}, AUC: {results[2]:.4f}")

    # Get predictions
    all_labels = []
    all_preds = []

    for images, labels in val_dataset:
        preds = model.predict(images)
        all_labels.extend(labels.numpy().flatten())
        all_preds.extend(preds.flatten())

    # Convert probabilities to binary predictions
    binary_preds = [1 if p >= 0.5 else 0 for p in all_preds]

    # Calculate metrics
    cm = confusion_matrix(all_labels, binary_preds)
    report = classification_report(all_labels, binary_preds, digits=4)
    auc_value = roc_auc_score(all_labels, all_preds)

    print(f"\nFold {fold_idx+1} - Confusion Matrix:")
    print(cm)
    print(f"\nFold {fold_idx+1} - Classification Report:")
    print(report)
    print(f"\nFold {fold_idx+1} - ROC AUC: {auc_value:.4f}")

    return {
        'cm': cm,
        'report': report,
        'auc': auc_value,
        'accuracy': results[1],
        'val_loss': results[0]
    }

def run_kfold_cv():
    neutrophil_files = [f for f in os.listdir(neutrophil_dir) if os.path.isfile(os.path.join(neutrophil_dir, f))]
    non_neutrophil_files = [f for f in os.listdir(non_neutrophil_dir) if os.path.isfile(os.path.join(non_neutrophil_dir, f))]

    print(f"Total Neutrophil samples: {len(neutrophil_files)}")
    print(f"Total NonNeutrophil samples: {len(non_neutrophil_files)}")

    # Create indices for all files
    all_indices = list(range(len(neutrophil_files) + len(non_neutrophil_files)))

    kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

    fold_results = []
    histories = []

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(all_indices)):
        print(f"\n{'='*50}")
        print(f"Starting Fold {fold_idx+1}/{NUM_FOLDS}")
        print(f"{'='*50}")

        train_dataset, val_dataset = prepare_fold_datasets(
            neutrophil_files, non_neutrophil_files, fold_idx, train_indices, val_indices
        )

        model, _ = create_model()

        history, model = train_model(model, train_dataset, val_dataset, fold_idx)

        fold_result = evaluate_model(model, val_dataset, fold_idx)
        fold_results.append(fold_result)
        histories.append(history)

        tf.keras.backend.clear_session()
        gc.collect()

    avg_accuracy = np.mean([res['accuracy'] for res in fold_results])
    avg_auc = np.mean([res['auc'] for res in fold_results])

    print("\n" + "="*50)
    print(f"K-Fold Cross-Validation Results (k={NUM_FOLDS})")
    print("="*50)
    print(f"Average Accuracy: {avg_accuracy:.4f}")
    print(f"Average AUC: {avg_auc:.4f}")
    print(f"Individual Fold Accuracies: {[res['accuracy'] for res in fold_results]}")
    print(f"Individual Fold AUCs: {[res['auc'] for res in fold_results]}")

    plot_learning_curves(histories)

    return fold_results, histories



def plot_learning_curves(histories):
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 1, 1)
    for i, history in enumerate(histories):
        acc = history.history['accuracy']
        val_acc = history.history['val_accuracy']
        epochs = range(1, len(acc) + 1)
        plt.plot(epochs, acc, '-', label=f'Train Acc Fold {i+1}', alpha=0.7)
        plt.plot(epochs, val_acc, '--', label=f'Val Acc Fold {i+1}', alpha=0.7)

    plt.title('Model Accuracy Across Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.subplot(2, 1, 2)
    for i, history in enumerate(histories):
        loss = history.history['loss']
        val_loss = history.history['val_loss']
        epochs = range(1, len(loss) + 1)
        plt.plot(epochs, loss, '-', label=f'Train Loss Fold {i+1}', alpha=0.7)
        plt.plot(epochs, val_loss, '--', label=f'Val Loss Fold {i+1}', alpha=0.7)

    plt.title('Model Loss Across Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig('kfold_learning_curves_partial_finetuning.png')
    plt.show()

if __name__ == "__main__":
    for split in ['train', 'val']:
        for cls in ['Neutrophil', 'NonNeutrophil']:
            os.makedirs(os.path.join(temp_dir, split, cls), exist_ok=True)

    try:
        fold_results, histories = run_kfold_cv()
    finally:
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)

: 